In [ ]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0)

@tool
def calculator(expression: str) -> str: # "20+10"
    """Calculate a mathematical expression."""
    try:
        result = eval(expression)  # 20+10
        return str(result)

    except Exception:
        return "Invalid mathematical expression"

tools = [calculator]

class AgentState(TypedDict):
    question: str
    decision: str
    tool_result: str
    answer: str

def agent(state: AgentState):

    question = state["question"]

    prompt = f"""
You are an AI Agent.
User Question:
{question}

Decide what to do.

If the question requires calculation,
respond exactly:

USE_TOOL

Otherwise respond exactly:

DIRECT_ANSWER
"""

    response = llm.invoke(prompt)

    decision = response.content.strip()

    return {
        "decision": decision
    }

def use_tool(state: AgentState):
    question = state["question"]

    prompt = f"""
Extract only the mathematical expression
from this question.

Question:
{question}

Example:
"What is 25 * 10?"
Return:
25 * 10
"""
    
    response = llm.invoke(prompt)

    expression = response.content.strip()

    result = calculator.invoke({
        "expression": expression
    })

    return {
        "tool_result": result
    }

def final_answer(state: AgentState):

    question = state["question"]
    tool_result = state.get("tool_result", "")

    if tool_result:
        prompt = f"""
Answer the user's question.

Question:
{question}

Calculator Result:
{tool_result}

Give only the final answer.
"""
    else:
        prompt = f"""
Answer this question clearly:

{question}
"""
    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

def router(state: AgentState):

    decision = state["decision"]

    if "USE_TOOL" in decision:
        return "tool"
    return "answer"

graph = StateGraph(AgentState) # it is prototype of agentic AI to get structured output
graph.add_node("agent", agent)
graph.add_node("tool", use_tool)
graph.add_node("answer", final_answer)

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    router,
    {
        "tool": "tool",
        "answer": "answer"
    }
)
graph.add_edge("tool", "answer")
graph.add_edge("answer", END)

app = graph.compile()

result = app.invoke({
    "question": "What is 10 + 20b?",
    "decision": "",
    "tool_result": "",
    "answer": ""
})
print("Agent Answer:")
print(result["answer"])

Agent Answer:
30b + 10
